In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

In [3]:
import pandas as pd 
import numpy as np
import joblib
from src.scripts.train import get_stratified_session_split

In [4]:
feature_data = pd.read_csv('../../data/processed/landlord/feature_data.csv')
error_data = pd.read_csv('../../data/processed/landlord/rf_test_errors.csv')

In [7]:
train_df , test_df = get_stratified_session_split(feature_data)

In [8]:
rf_model = joblib.load("../..//models/landlord/rf_model.joblib")

In [9]:
import time
import numpy as np

def estimate_prediction_latency(model, X_sample, iterations=1000):
    """
    Estimates the average time (in milliseconds) to predict a single sample.
    """
    # Ensure X_sample is the correct shape (1, n_features)
    sample = X_sample.iloc[[0]] if hasattr(X_sample, 'iloc') else X_sample[[0]]
    
    # 1. Warm-up (prevents first-run overhead from skewing results)
    for _ in range(10):
        _ = model.predict(sample)
    
    # 2. Benchmark loop
    start_time = time.perf_counter()
    for _ in range(iterations):
        _ = model.predict(sample)
    end_time = time.perf_counter()
    
    # 3. Calculation
    total_time = end_time - start_time
    avg_latency_ms = (total_time / iterations) * 1000
    
    print(f"--- Prediction Latency Analysis ---")
    print(f"Total iterations: {iterations}")
    print(f"Avg Latency:      {avg_latency_ms:.4f} ms")
    print(f"Predictions/sec:  {1000 / avg_latency_ms:.2f}")
    
    return avg_latency_ms

# Usage:
# sample_input = X_test.iloc[[0]] 
# avg_ms = estimate_prediction_latency(best_rf, sample_input)

In [10]:
sample_input = test_df[rf_model.feature_names_in_].iloc[[3]] 
avg_ms = estimate_prediction_latency(rf_model, sample_input)

--- Prediction Latency Analysis ---
Total iterations: 1000
Avg Latency:      2.0873 ms
Predictions/sec:  479.10
